# 1. Objective and Scope

## 1.1 Objective

The objective of this notebook is to transform the validated document page records produced by the existing ingestion pipeline into smaller, coherent, traceable text chunks suitable for subsequent embedding and retrieval.

The implementation will explore and compare two chunking strategies:

* **Fixed-Window Chunking:** Divides page content into consecutive character-based windows with configurable overlap.
* **Structure-Aware Recursive Chunking:** Splits content using meaningful document boundaries, prioritizing Markdown sections, paragraphs, lines, and sentences.

## 1.2 Existing Infrastructure

This project already provides a document ingestion layer responsible for:

* Document discovery and text extraction.
* Document and page schema validation.
* Document identification and content hashing.
* Duplicate and unchanged-document detection.
* JSONL persistence.

These capabilities will be reused without reimplementation.

The chunking process will consume the existing:

* `DocumentRecord` records from `documents.jsonl`.
* `PageRecord` records from `pages.jsonl`.

Raw source files will not be rediscovered or re-extracted.

## 1.3 Scope

This notebook covers:

1. Loading and auditing existing processed records.
2. Defining the chunk data contract and configuration.
3. Implementing fixed-window chunking.
4. Parsing Markdown document structure.
5. Implementing structure-aware recursive chunking.
6. Preserving source provenance and character offsets.
7. Generating deterministic chunk identifiers.
8. Comparing chunking strategies.
9. Persisting and validating chunk records.
10. Evaluating implementation correctness through automated tests.

## 1.4 Initial Configuration

The initial experimental configuration is:

| Parameter         |            Initial Value |
| ----------------- | -----------------------: |
| Target Chunk Size |         1,200 characters |
| Target Overlap    |           180 characters |
| Chunk Length Unit | Python string characters |
| Input Unit        |    Individual PageRecord |
| Output Format     |                    JSONL |

These values are provisional experimental settings. They are not assumed to be optimal and will require validation against the selected embedding model's tokenizer.

## 1.5 Scope Exclusions

The following capabilities are outside the scope of this notebook:

* Document discovery and extraction.
* PDF parsing.
* Embedding generation.
* Vector database construction.
* Similarity search and retrieval.
* Reranking and query rewriting.
* LLM-based question answering.

## 1.6 Expected Output

The primary output is:

`data/knowledge_base/processed/chunks.jsonl`

Every generated chunk must retain its document identity, page number, source character offsets, content hash, chunking strategy, and configuration metadata.

The final objective is to produce reproducible, validated, source-traceable chunks ready for the embedding stage.


# 2. Load Existing Processed Records

## 2.1 Objective

Load the previously ingested document and page records from the existing JSONL storage.

This step reuses the ingestion module's existing loading functions and Pydantic models. No document discovery, extraction, or ingestion operations are performed.

## 2.2 Input Files

* `data/knowledge_base/processed/documents.jsonl`
* `data/knowledge_base/processed/pages.jsonl`

## 2.3 Loading Strategy

The existing functions are responsible for reading JSONL records and validating them against their corresponding Pydantic models:

* `load_existing_documents()`
* `load_existing_pages()`

The resulting objects will be used directly by the chunking experiments.

## 2.4 Expected Data Structures

* Documents: A dictionary indexed by `document_id`.
* Pages: A list of `PageRecord` objects.

## 2.5 Validation Objective

Confirm that both datasets are successfully loaded and that the returned objects use the expected model types.


In [1]:
import sys
from pathlib import Path

# Locate the project root from the current working directory.
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "app" / "ingestion").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Expected an 'app/ingestion' directory."
    )

# Add the project root to Python's module search path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import existing ingestion models and loaders.
from app.ingestion.models import DocumentRecord, PageRecord
from app.ingestion.pipeline import (
    load_existing_documents,
    load_existing_pages,
)

# Define processed data paths.
PROCESSED_DIR = PROJECT_ROOT / "data" / "knowledge_base" / "processed"

DOCUMENTS_PATH = PROCESSED_DIR / "documents.jsonl"
PAGES_PATH = PROCESSED_DIR / "pages.jsonl"

# Load existing records.
documents = load_existing_documents(DOCUMENTS_PATH)
pages = load_existing_pages(PAGES_PATH)

# Display loading results.
print("PROJECT CONFIGURATION")
print("=" * 50)
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {CURRENT_DIR}")

print("\nDOCUMENT LOADING")
print("=" * 50)
print(f"Documents loaded: {len(documents)}")
print(f"Pages loaded: {len(pages)}")

print("\nMODEL VALIDATION")
print("=" * 50)
print(
    "All documents are DocumentRecord:",
    all(isinstance(record, DocumentRecord) for record in documents.values()),
)
print(
    "All pages are PageRecord:",
    all(isinstance(record, PageRecord) for record in pages),
)

print("\nINPUT FILES")
print("=" * 50)
print(f"Documents: {DOCUMENTS_PATH}")
print(f"Pages: {PAGES_PATH}")

PROJECT CONFIGURATION
Project root: d:\Projects\Local Mini RAG Document QA System
Working directory: d:\Projects\Local Mini RAG Document QA System\notebooks

DOCUMENT LOADING
Documents loaded: 6
Pages loaded: 6

MODEL VALIDATION
All documents are DocumentRecord: True
All pages are PageRecord: True

INPUT FILES
Documents: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\documents.jsonl
Pages: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\pages.jsonl


### Result Interpretation — Loading Existing Records

The processed ingestion records were loaded successfully.

| Metric                    | Result |
| ------------------------- | -----: |
| Documents loaded          |      6 |
| Pages loaded              |      6 |
| DocumentRecord validation | Passed |
| PageRecord validation     | Passed |

The notebook successfully located the project root and imported the existing ingestion modules.

**Finding:** The chunking stage can reuse the existing processed records directly. Repeating document discovery or text extraction is unnecessary.

**Next Step:** Validate the integrity, relationships, and text characteristics of the loaded corpus before implementing chunking strategies.


# 3. Input Validation and Corpus Audit

## Objective

Validate the integrity and consistency of the previously processed document and page records.

This audit examines:

1. Document identifiers and content integrity.
2. Page identifiers and document references.
3. Empty or invalid text records.
4. Page numbering and document relationships.
5. Corpus-level text statistics.

## Validation Principles

* Every document must have a unique identifier.
* Every page must reference an existing document.
* Page content must be a valid string.
* Empty pages must be identified.
* Page numbering must be inspected before defining chunk boundaries.
* All statistics must be calculated from the actual processed corpus.

No document extraction or text transformation will be performed during this stage.

In [2]:
from collections import Counter
from statistics import mean

# --------------------------------------------------
# 1. Document Integrity
# --------------------------------------------------

document_ids = [document.document_id for document in documents.values()]

document_id_counts = Counter(document_ids)

duplicate_document_ids = [
    document_id for document_id, count in document_id_counts.items() if count > 1
]

empty_document_content = [
    document.document_id
    for document in documents.values()
    if not document.content.strip()
]

# --------------------------------------------------
# 2. Page Integrity
# --------------------------------------------------

page_document_ids = [page.document_id for page in pages]

page_id_counts = Counter((page.document_id, page.page_number) for page in pages)

duplicate_pages = [page_id for page_id, count in page_id_counts.items() if count > 1]

missing_document_references = [
    page.document_id for page in pages if page.document_id not in documents
]

empty_pages = [
    (page.document_id, page.page_number) for page in pages if not page.content.strip()
]

# --------------------------------------------------
# 3. Page Numbering
# --------------------------------------------------

pages_by_document = {}

for page in pages:
    pages_by_document.setdefault(page.document_id, []).append(page.page_number)

page_numbering = {}

for document_id, page_numbers in pages_by_document.items():
    sorted_numbers = sorted(page_numbers)

    page_numbering[document_id] = {
        "page_numbers": sorted_numbers,
        "is_sorted_contiguous": (
            sorted_numbers
            == list(
                range(
                    sorted_numbers[0],
                    sorted_numbers[0] + len(sorted_numbers),
                )
            )
        ),
    }

# --------------------------------------------------
# 4. Text Statistics
# --------------------------------------------------

page_lengths = [len(page.content) for page in pages]

document_lengths = [len(document.content) for document in documents.values()]

total_page_characters = sum(page_lengths)
total_document_characters = sum(document_lengths)

# --------------------------------------------------
# 5. Document-to-Page Relationships
# --------------------------------------------------

pages_per_document = Counter(page.document_id for page in pages)

documents_without_pages = [
    document_id for document_id in documents if pages_per_document[document_id] == 0
]

# --------------------------------------------------
# 6. Audit Report
# --------------------------------------------------

print("DOCUMENT INTEGRITY")
print("=" * 55)
print(f"Unique document IDs: {len(set(document_ids))}")
print(f"Duplicate document IDs: {duplicate_document_ids}")
print(f"Documents with empty content: {empty_document_content}")

print("\nPAGE INTEGRITY")
print("=" * 55)
print(f"Total pages: {len(pages)}")
print(f"Duplicate pages: {duplicate_pages}")
print(f"Missing document references: {missing_document_references}")
print(f"Empty pages: {empty_pages}")

print("\nPAGE NUMBERING")
print("=" * 55)

for document_id, result in page_numbering.items():
    print(f"{document_id}: {result}")

print("\nDOCUMENT-PAGE RELATIONSHIPS")
print("=" * 55)
print(f"Documents without pages: {documents_without_pages}")

for document_id, count in pages_per_document.items():
    print(f"{document_id}: {count} page(s)")

print("\nCORPUS STATISTICS")
print("=" * 55)
print(f"Total document characters: {total_document_characters:,}")
print(f"Total page characters: {total_page_characters:,}")
print(f"Minimum page length: {min(page_lengths):,}")
print(f"Maximum page length: {max(page_lengths):,}")
print(f"Average page length: {mean(page_lengths):,.2f}")
print(f"Minimum document length: {min(document_lengths):,}")
print(f"Maximum document length: {max(document_lengths):,}")
print(f"Average document length: {mean(document_lengths):,.2f}")

DOCUMENT INTEGRITY
Unique document IDs: 6
Duplicate document IDs: []
Documents with empty content: []

PAGE INTEGRITY
Total pages: 6
Duplicate pages: []
Missing document references: []
Empty pages: []

PAGE NUMBERING
5eaa2f3a4137bbd8: {'page_numbers': [1], 'is_sorted_contiguous': True}
4bc29707830b7f97: {'page_numbers': [1], 'is_sorted_contiguous': True}
6e0ae6c5f11c01b6: {'page_numbers': [1], 'is_sorted_contiguous': True}
1f8658d7b4d1d931: {'page_numbers': [1], 'is_sorted_contiguous': True}
17f4f629bc6c06aa: {'page_numbers': [1], 'is_sorted_contiguous': True}
b8c040cc92d5243c: {'page_numbers': [1], 'is_sorted_contiguous': True}

DOCUMENT-PAGE RELATIONSHIPS
Documents without pages: []
5eaa2f3a4137bbd8: 1 page(s)
4bc29707830b7f97: 1 page(s)
6e0ae6c5f11c01b6: 1 page(s)
1f8658d7b4d1d931: 1 page(s)
17f4f629bc6c06aa: 1 page(s)
b8c040cc92d5243c: 1 page(s)

CORPUS STATISTICS
Total document characters: 118,204
Total page characters: 118,204
Minimum page length: 9,497
Maximum page length: 32,36

### Result Interpretation — Input Validation and Corpus Audit

The corpus passed all integrity checks.

| Metric                      |    Result |
| --------------------------- | --------: |
| Documents                   |         6 |
| Pages                       |         6 |
| Total characters            |   118,204 |
| Minimum page length         |     9,497 |
| Maximum page length         |    32,360 |
| Average page length         | 19,700.67 |
| Duplicate document IDs      |         0 |
| Duplicate pages             |         0 |
| Missing document references |         0 |
| Empty documents or pages    |         0 |

### Structural Findings

* Each document currently contains exactly one logical page.
* All page numbers start at 1.
* Every page references an existing document.
* Document content and page content have matching total character counts.

### Technical Findings

1. The corpus is structurally valid and ready for chunking experiments.
2. Chunk provenance must preserve document IDs and page numbers.
3. Document length varies significantly, so chunking behavior must be evaluated across different input sizes.
4. Multi-page document handling should remain supported by the implementation, even though the current corpus does not exercise that case.

**Conclusion:** No ingestion or data-integrity issues were detected. The existing processed records can be used as the input for the chunking pipeline.

**Next Step:** Inspect Markdown structure to understand headings, paragraphs, lists, tables, and code blocks before selecting chunk boundaries.


# 4. Understanding Document Structure

## Objective

Inspect the actual Markdown structure of the processed corpus before implementing chunking algorithms.

## Structural Elements

The audit focuses on:

1. ATX headings (`#`, `##`, `###`, etc.).
2. Paragraphs.
3. Unordered and ordered lists.
4. Markdown tables.
5. Fenced code blocks using backticks or tildes.
6. Horizontal rules.

## Why Structure Matters

Chunk boundaries should preserve meaningful relationships between document elements.

For example, a heading should remain associated with its corresponding content, list items should preferably remain intact, and code blocks should not be interpreted as ordinary Markdown headings.

## Scope

This section only inspects and reports document structure. It does not modify the source text or implement chunking.


In [ ]:
import re
from collections import Counter

# Markdown structure patterns.
PATTERNS = {
    "headings": re.compile(r"^\s{0,3}#{1,6}\s+\S", re.MULTILINE),
    "unordered_lists": re.compile(r"^\s*[-+*]\s+\S", re.MULTILINE),
    "ordered_lists": re.compile(r"^\s*\d+[.)]\s+\S", re.MULTILINE),
    "table_separator": re.compile(
        r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$",
        re.MULTILINE,
    ),
    "horizontal_rules": re.compile(
        r"^\s{0,3}((\*\s*){3,}|(-\s*){3,}|(_\s*){3,})$",
        re.MULTILINE,
    ),
}


def inspect_markdown_structure(content: str) -> dict:
    lines = content.splitlines()

    fenced_code_blocks = 0
    inside_code_block = False
    fence_marker = None

    code_lines = 0
    structural_counts = Counter()

    for line in lines:
        stripped = line.lstrip()

        # Detect fenced code block boundaries.
        fence_match = re.match(r"^(`{3,}|~{3,})", stripped)

        if fence_match:
            marker = fence_match.group(1)

            if not inside_code_block:
                inside_code_block = True
                fence_marker = marker[0]
                fenced_code_blocks += 1
                continue

            if marker[0] == fence_marker:
                inside_code_block = False
                fence_marker = None
                continue

        if inside_code_block:
            code_lines += 1
            continue

        if not line.strip():
            continue

        if PATTERNS["headings"].match(line):
            structural_counts["headings"] += 1

        if PATTERNS["unordered_lists"].match(line):
            structural_counts["unordered_lists"] += 1

        if PATTERNS["ordered_lists"].match(line):
            structural_counts["ordered_lists"] += 1

        if "|" in line and line.count("|") >= 2:
            structural_counts["table_like_lines"] += 1

        if PATTERNS["table_separator"].match(line):
            structural_counts["table_separator_lines"] += 1

        if PATTERNS["horizontal_rules"].match(line):
            structural_counts["horizontal_rules"] += 1

    return {
        "total_lines": len(lines),
        "non_empty_lines": sum(bool(line.strip()) for line in lines),
        "fenced_code_blocks": fenced_code_blocks,
        "code_lines": code_lines,
        "unclosed_code_fence": inside_code_block,
        **dict(structural_counts),
    }


# Inspect all actual pages.
structure_reports = []

for page in pages:
    report = inspect_markdown_structure(page.content)

    structure_reports.append(
        {
            "document_id": page.document_id,
            "page_number": page.page_number,
            **report,
        }
    )


# Display per-document results.
print("MARKDOWN STRUCTURE AUDIT")
print("=" * 70)

for report in structure_reports:
    print(f"\nDocument: {report['document_id']}")
    print(f"Page: {report['page_number']}")

    for key, value in report.items():
        if key not in {"document_id", "page_number"}:
            print(f"{key}: {value}")


# Aggregate structural statistics.
aggregate = Counter()

for report in structure_reports:
    for key, value in report.items():
        if key not in {"document_id", "page_number"}:
            if isinstance(value, int):
                aggregate[key] += value

print("\nCORPUS STRUCTURE SUMMARY")
print("=" * 70)

for key, value in aggregate.items():
    print(f"{key}: {value}")

MARKDOWN STRUCTURE AUDIT

Document: 5eaa2f3a4137bbd8
Page: 1
total_lines: 374
non_empty_lines: 257
fenced_code_blocks: 6
code_lines: 42
unclosed_code_fence: False
headings: 30
unordered_lists: 52
table_like_lines: 51
table_separator_lines: 5
ordered_lists: 3

Document: 4bc29707830b7f97
Page: 1
total_lines: 259
non_empty_lines: 186
fenced_code_blocks: 2
code_lines: 2
unclosed_code_fence: False
headings: 22
unordered_lists: 28
horizontal_rules: 2
table_like_lines: 21
table_separator_lines: 3
ordered_lists: 13

Document: 6e0ae6c5f11c01b6
Page: 1
total_lines: 623
non_empty_lines: 431
fenced_code_blocks: 10
code_lines: 56
unclosed_code_fence: False
headings: 65
horizontal_rules: 1
unordered_lists: 50
ordered_lists: 49
table_like_lines: 100
table_separator_lines: 9

Document: 1f8658d7b4d1d931
Page: 1
total_lines: 565
non_empty_lines: 344
fenced_code_blocks: 0
code_lines: 0
unclosed_code_fence: False
headings: 45
unordered_lists: 75
horizontal_rules: 15
table_like_lines: 63
table_separator_li

### Result Interpretation — Markdown Structure Audit

The Markdown structure audit identified a highly structured corpus.

| Structural Element    | Count |
| --------------------- | ----: |
| Headings              |   205 |
| Unordered list items  |   314 |
| Ordered list items    |    86 |
| Table-like lines      |   340 |
| Table separator lines |    31 |
| Fenced code blocks    |    34 |
| Code lines            |   159 |
| Horizontal rules      |    18 |
| Total lines           | 2,368 |
| Non-empty lines       | 1,635 |

### Technical Findings

1. The corpus contains a substantial heading hierarchy that can support section-aware chunk metadata.
2. Lists represent a significant portion of the content and should be preserved as logical units whenever possible.
3. Table detection requires more precise parsing than counting pipe characters.
4. Code fences must be tracked to prevent code content from being misinterpreted as Markdown structure.
5. All inspected documents have balanced fenced code blocks according to the initial structural audit.

### Implementation Implications

The chunking implementation should:

* Maintain heading hierarchy and section paths.
* Preserve paragraph boundaries.
* Respect ordered and unordered list boundaries.
* Recognize Markdown tables using structural validation.
* Support backtick and tilde code fences.
* Split oversized structural elements without losing text or provenance.

**Conclusion:** The corpus requires structure-aware chunking in addition to a fixed-window baseline. The structural audit provides the evidence needed to design the Markdown parser.

**Next Step:** Define chunking fundamentals, configuration parameters, and measurable constraints before implementing the first strategy.


# 5. Chunking Fundamentals and Configuration

## Objective

Define the initial chunking configuration and evaluation criteria before implementing any chunking strategy.

## Chunking Parameters

| Parameter            | Initial Value | Description                               |
| -------------------- | ------------: | ----------------------------------------- |
| CHUNK_SIZE_CHARS     |          1200 | Target chunk length in Unicode characters |
| CHUNK_OVERLAP_CHARS  |           180 | Target overlap between consecutive chunks |
| MIN_CHUNK_SIZE_CHARS |           100 | Initial minimum size for ordinary chunks  |
| CHUNKING_VERSION     |             1 | Chunking algorithm version                |

These values are experimental starting points, not proven optimal settings.

## Measurement Criteria

The experiments will evaluate:

1. Total number of generated chunks.
2. Average, minimum, and maximum chunk lengths.
3. Actual overlap between consecutive chunks.
4. Oversized chunk frequency.
5. Source text coverage and integrity.
6. Preservation of structural boundaries.
7. Deterministic output across repeated runs.

## Important Constraints

* Character counts use Python Unicode string indexing.
* Character counts are not equivalent to tokenizer token counts.
* Chunk content must preserve its exact source substring.
* Chunk offsets must refer to the original page content.
* Structure-aware splitting may produce chunks that exceed the target size when preserving a logical unit is preferable.
* No embedding or retrieval evaluation is performed in this stage.

## Experimental Principle

The fixed-window strategy provides a baseline. The structure-aware recursive strategy will be compared against it using the same corpus and measurable integrity criteria.


In [ ]:
# Initial chunking configuration.

CHUNK_SIZE_CHARS = 1200
CHUNK_OVERLAP_CHARS = 180
MIN_CHUNK_SIZE_CHARS = 100
CHUNKING_VERSION = 1

# Derived configuration.
EFFECTIVE_STEP_CHARS = CHUNK_SIZE_CHARS - CHUNK_OVERLAP_CHARS

# Configuration validation.
assert CHUNK_SIZE_CHARS > 0
assert 0 <= CHUNK_OVERLAP_CHARS < CHUNK_SIZE_CHARS
assert 0 < MIN_CHUNK_SIZE_CHARS <= CHUNK_SIZE_CHARS

# Display configuration.
print("CHUNKING CONFIGURATION")
print("=" * 55)
print(f"Target chunk size: {CHUNK_SIZE_CHARS:,} characters")
print(f"Target overlap: {CHUNK_OVERLAP_CHARS:,} characters")
print(f"Effective step: {EFFECTIVE_STEP_CHARS:,} characters")
print(f"Minimum chunk size: {MIN_CHUNK_SIZE_CHARS:,} characters")
print(f"Chunking version: {CHUNKING_VERSION}")

print("\nCONFIGURATION VALIDATION")
print("=" * 55)
print("Chunk size is positive: True")
print("Overlap is smaller than chunk size: True")
print("Minimum size is valid: True")

CHUNKING CONFIGURATION
Target chunk size: 1,200 characters
Target overlap: 180 characters
Effective step: 1,020 characters
Minimum chunk size: 100 characters
Chunking version: 1

CONFIGURATION VALIDATION
Chunk size is positive: True
Overlap is smaller than chunk size: True
Minimum size is valid: True


### Result Interpretation — Chunking Configuration

The initial chunking configuration was defined and validated successfully.

| Parameter          |            Value |
| ------------------ | ---------------: |
| Target chunk size  | 1,200 characters |
| Target overlap     |   180 characters |
| Effective step     | 1,020 characters |
| Minimum chunk size |   100 characters |
| Chunking version   |                1 |

All configuration assertions passed.

### Technical Findings

1. The configuration uses character-based measurements.
2. The fixed-window strategy will use a 1,020-character step.
3. The overlap value is a target for the fixed-window baseline.
4. Structure-aware chunking may produce different actual chunk sizes and overlaps to preserve logical boundaries.
5. These values are experimental starting points and require evaluation.

**Conclusion:** The initial configuration is valid and ready to be used by the chunking implementation.

**Next Step:** Define a reusable ChunkRecord schema that preserves source content, offsets, structural metadata, and provenance.
